# Predicting Heart Disease Risk Using Machine Learning Models
### Framingham Heart Study Dataset
**Team:** Aryan Shah | Elyas Khoubach | Matias Elban Stringa | Vin Patel

---

This notebook contains the full pipeline for all four models:
- **Section 1** — Imports and Setup
- **Section 2** — Load Data
- **Section 3** — Exploratory Data Analysis (EDA)
- **Section 4** — Preprocessing
- **Section 5** — Logistic Regression *(Vin)*
- **Section 6** — Random Forest *(Aryan)*
- **Section 7** — SVM *(Elyas)*
- **Section 8** — XGBoost *(Matias)*
- **Section 9** — Final Model Comparison

---
## Section 1 — Imports and Setup
We import every library we need for the entire notebook here at the top.
If you get a ModuleNotFoundError, run:  `pip install <library_name>`

In [ ]:
# ── Core data libraries ──────────────────────────────────────────────────────
import pandas as pd          # for loading and manipulating the dataset as a table
import numpy as np           # for numerical operations
import warnings
warnings.filterwarnings('ignore')  # suppress minor warnings to keep output clean

# ── Visualization libraries ──────────────────────────────────────────────────
import matplotlib.pyplot as plt    # for plotting charts and graphs
import seaborn as sns              # for prettier statistical plots
sns.set_style('whitegrid')         # clean white background for all plots

# ── Preprocessing ────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split   # splits data into train/val/test
from sklearn.preprocessing import MinMaxScaler          # scales features to [0, 1]
from imblearn.over_sampling import SMOTE                # handles class imbalance

# ── Models ───────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression     # Vin's model
from sklearn.ensemble import RandomForestClassifier     # Aryan's model
from sklearn.svm import SVC                             # Elyas's model
import xgboost as xgb                                   # Matias's model

# ── Hyperparameter tuning ────────────────────────────────────────────────────
from sklearn.model_selection import GridSearchCV        # tries all combinations of settings

# ── Evaluation metrics ───────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,           # overall correct predictions
    classification_report,    # full breakdown of precision, recall, f1
    confusion_matrix,         # 2x2 table of correct/incorrect predictions
    roc_auc_score,            # area under the ROC curve (our main metric)
    roc_curve,                # plots the ROC curve
    f1_score,                 # balance between precision and recall
    recall_score,             # sensitivity — how many sick patients we caught
    precision_score           # how many of our positive predictions were correct
)

# ── SHAP (for XGBoost explainability) ───────────────────────────────────────
import shap                   # explains individual predictions from XGBoost

print('All libraries loaded successfully!')

---
## Section 2 — Load Data
We load the CSV file and take a first look at what the data contains.

In [ ]:
# Load the dataset from the CSV file
# Make sure framingham.csv is in the same folder as this notebook
df = pd.read_csv('framingham.csv')

# Show the first 5 rows so we can see what the data looks like
print('Dataset shape:', df.shape)  # (rows, columns)
df.head()

In [ ]:
# Show information about each column:
# - column name
# - how many non-null values it has (missing values show up here)
# - the data type (int, float, object)
df.info()

In [ ]:
# Show basic statistics for every column:
# mean, min, max, standard deviation, quartiles
# This helps us spot anything weird like impossible values
df.describe()

---
## Section 3 — Exploratory Data Analysis (EDA)
Before building any model, we need to understand the data:
- How many missing values are there?
- How imbalanced is our target variable?
- What do the feature distributions look like?

In [ ]:
# ── 3.1 Missing Values ───────────────────────────────────────────────────────

# Count missing values per column
missing = df.isnull().sum()

# Calculate what percentage of each column is missing
missing_pct = (missing / len(df)) * 100

# Combine into a readable table
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
})

# Only show columns that actually have missing values
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('Columns with missing values:')
print(missing_df)

In [ ]:
# Visualize missing values as a bar chart
plt.figure(figsize=(10, 4))
sns.barplot(x=missing_df.index, y=missing_df['Missing %'], color='steelblue')
plt.title('Percentage of Missing Values per Column')
plt.ylabel('Missing %')
plt.xlabel('Column')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.2 Class Imbalance ──────────────────────────────────────────────────────

# Count how many patients have and don't have CHD
# TenYearCHD is our target: 1 = developed heart disease, 0 = did not
class_counts = df['TenYearCHD'].value_counts()
class_pct = df['TenYearCHD'].value_counts(normalize=True) * 100

print('Class distribution:')
print(f'  No CHD (0): {class_counts[0]} patients ({class_pct[0]:.1f}%)')
print(f'  CHD    (1): {class_counts[1]} patients ({class_pct[1]:.1f}%)')

# Visualize the imbalance
plt.figure(figsize=(6, 4))
sns.countplot(x='TenYearCHD', data=df, palette=['steelblue', 'tomato'])
plt.title('Class Imbalance: CHD vs No CHD')
plt.xlabel('TenYearCHD (0 = No, 1 = Yes)')
plt.ylabel('Patient Count')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Feature Distributions ────────────────────────────────────────────────

# Plot a histogram for every column to see the distribution of values
# This helps us spot skewed distributions or outliers
df.hist(figsize=(16, 12), bins=30, color='steelblue', edgecolor='white')
plt.suptitle('Distribution of All Features', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Correlation Heatmap ──────────────────────────────────────────────────

# A correlation matrix shows how much each feature is related to every other feature
# Values close to 1 or -1 mean strong relationship
# Values close to 0 mean no relationship
plt.figure(figsize=(12, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

---
## Section 4 — Preprocessing
We clean and prepare the data before feeding it into any model.

Steps:
1. Handle missing values
2. Separate features from target
3. Normalize features
4. Split into train / validation / test sets
5. Apply SMOTE to fix class imbalance (on training set only)

In [ ]:
# ── Step 1: Handle Missing Values ────────────────────────────────────────────

# Make a copy so we don't accidentally change the original loaded data
df_clean = df.copy()

# For each column, fill missing values with the median of that column
# We use median instead of mean because median is not affected by extreme outliers
# For example if most glucose values are around 80 but one person has 300,
# the median won't be pulled toward 300 the way the mean would be
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        if df_clean[col].dtype in ['float64', 'int64']:
            # Continuous feature — use median
            median_val = df_clean[col].median()
            df_clean[col].fillna(median_val, inplace=True)
            print(f'  Filled {col} missing values with median = {median_val:.2f}')
        else:
            # Categorical feature — use mode (most common value)
            mode_val = df_clean[col].mode()[0]
            df_clean[col].fillna(mode_val, inplace=True)
            print(f'  Filled {col} missing values with mode = {mode_val}')

# Confirm no missing values remain
print(f'\nTotal missing values after imputation: {df_clean.isnull().sum().sum()}')

In [ ]:
# ── Step 2: Separate Features from Target ────────────────────────────────────

# X = all input features (everything except the target column)
# y = the target column we are trying to predict
X = df_clean.drop('TenYearCHD', axis=1)   # axis=1 means drop a column (not a row)
y = df_clean['TenYearCHD']

print('Features (X) shape:', X.shape)     # should be (rows, 15)
print('Target (y) shape:', y.shape)       # should be (rows,)
print('\nFeature columns:')
print(list(X.columns))

In [ ]:
# ── Step 3: Train / Validation / Test Split ──────────────────────────────────

# We split the data three ways:
# - Training set (70%): used to train all models
# - Validation set (15%): used during development to tune hyperparameters
# - Test set (15%): held out completely, only used once at the very end
#
# stratify=y means the 85/15 CHD class ratio is preserved in all three splits
# random_state=42 means the split is reproducible (same result every time)

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: split the 30% temp into 15% val and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f'Training set:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'Test set:       {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nClass balance in training set:')
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
# ── Step 4: Normalize Features ───────────────────────────────────────────────

# MinMaxScaler scales every feature to be between 0 and 1
# This is important for Logistic Regression and SVM which are sensitive to scale
# For example, age ranges from 30-70 but cigsPerDay ranges from 0-70
# Without scaling, the model might give more weight to features with larger numbers
#
# IMPORTANT: we fit the scaler ONLY on training data
# then apply (transform) it to validation and test
# This prevents information from val/test leaking into training

scaler = MinMaxScaler()

# Fit on training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Only transform (do not fit) validation and test
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Scaling complete.')
print(f'Feature range after scaling — Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}')

In [ ]:
# ── Step 5: Apply SMOTE to Training Set ──────────────────────────────────────

# SMOTE stands for Synthetic Minority Over-sampling Technique
# The problem: only ~15% of patients in our data have CHD
# If we train on this imbalanced data, the model learns to mostly predict 'no CHD'
# and still gets 85% accuracy — but misses all the sick patients
#
# SMOTE fixes this by creating new SYNTHETIC CHD-positive patients
# It does this by finding existing CHD patients and interpolating between them
# to create realistic new examples, not just copying existing ones
#
# CRITICAL: we ONLY apply SMOTE to the training set
# Validation and test sets must remain original so evaluation is realistic

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print('Before SMOTE:')
print(f'  CHD=0: {(y_train == 0).sum()}  CHD=1: {(y_train == 1).sum()}')
print('\nAfter SMOTE:')
print(f'  CHD=0: {(y_train_resampled == 0).sum()}  CHD=1: {(y_train_resampled == 1).sum()}')
print(f'\nTotal training samples after SMOTE: {len(X_train_resampled)}')

---
## Section 5 — Logistic Regression *(Vin Patel)*
Our first baseline model. Linear, fast, and interpretable.
The odds ratios tell us exactly how much each feature increases or decreases CHD risk.

In [ ]:
# ── 5.1 Train Logistic Regression ────────────────────────────────────────────

# penalty='l2' means L2 regularization — prevents overfitting by penalizing
#   large coefficient values
# C=1.0 controls regularization strength — lower C = stronger regularization
# class_weight='balanced' automatically adjusts weights so the model
#   pays more attention to the minority class (CHD=1 patients)
# solver='liblinear' is recommended for small-to-medium datasets
# max_iter=1000 gives the model enough iterations to converge

lr_model = LogisticRegression(
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    solver='liblinear',
    max_iter=1000,
    random_state=42
)

# Train the model on the SMOTE-resampled training data
lr_model.fit(X_train_resampled, y_train_resampled)
print('Logistic Regression trained successfully.')

In [ ]:
# ── 5.2 Evaluate on Validation Set ───────────────────────────────────────────

# predict() gives us hard class labels (0 or 1)
lr_val_preds = lr_model.predict(X_val_scaled)

# predict_proba() gives us probabilities — we need this for AUC-ROC
# [:,1] means take the probability of the positive class (CHD=1)
lr_val_proba = lr_model.predict_proba(X_val_scaled)[:, 1]

# Calculate all metrics
lr_val_accuracy  = accuracy_score(y_val, lr_val_preds)
lr_val_auc       = roc_auc_score(y_val, lr_val_proba)
lr_val_f1        = f1_score(y_val, lr_val_preds)
lr_val_recall    = recall_score(y_val, lr_val_preds)    # sensitivity
lr_val_precision = precision_score(y_val, lr_val_preds)

print('=== Logistic Regression — Validation Set ===')
print(f'  Accuracy:  {lr_val_accuracy:.4f}')
print(f'  AUC-ROC:   {lr_val_auc:.4f}')
print(f'  F1 Score:  {lr_val_f1:.4f}')
print(f'  Recall:    {lr_val_recall:.4f}  (sensitivity — % of sick patients caught)')
print(f'  Precision: {lr_val_precision:.4f}')
print()
print('Full classification report:')
print(classification_report(y_val, lr_val_preds, target_names=['No CHD', 'CHD']))

In [ ]:
# ── 5.3 Confusion Matrix ─────────────────────────────────────────────────────

# The confusion matrix shows a 2x2 table:
#   Top-left:     True Negatives  (predicted No CHD, actually No CHD) ✓
#   Top-right:    False Positives (predicted CHD, actually No CHD) ✗
#   Bottom-left:  False Negatives (predicted No CHD, actually CHD) ✗ ← most dangerous
#   Bottom-right: True Positives  (predicted CHD, actually CHD) ✓

cm_lr = confusion_matrix(y_val, lr_val_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred No CHD', 'Pred CHD'],
            yticklabels=['Actual No CHD', 'Actual CHD'])
plt.title('Logistic Regression — Confusion Matrix (Validation)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.4 ROC Curve ────────────────────────────────────────────────────────────

# roc_curve returns three arrays:
# fpr = false positive rate at each threshold
# tpr = true positive rate (sensitivity) at each threshold
# thresholds = the decision thresholds used

fpr_lr, tpr_lr, _ = roc_curve(y_val, lr_val_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr_lr, tpr_lr, color='steelblue', lw=2,
         label=f'Logistic Regression (AUC = {lr_val_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Logistic Regression — ROC Curve (Validation)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.5 Odds Ratios ──────────────────────────────────────────────────────────

# In Logistic Regression, each feature gets a coefficient
# The odds ratio = e^coefficient
# Odds ratio > 1 means this feature INCREASES CHD risk
# Odds ratio < 1 means this feature DECREASES CHD risk
# Example: odds ratio of 1.5 for age means each 1-unit increase in
# (scaled) age multiplies the odds of CHD by 1.5

feature_names = X.columns.tolist()
coefficients = lr_model.coef_[0]           # raw coefficients from the model
odds_ratios = np.exp(coefficients)          # convert to odds ratios using e^

# Create a sorted table from most to least influential
odds_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Odds Ratio': odds_ratios
}).sort_values('Odds Ratio', ascending=False)

print('Logistic Regression — Odds Ratios (sorted by importance):')
print(odds_df.to_string(index=False))

# Visualize as a horizontal bar chart
plt.figure(figsize=(8, 6))
colors = ['tomato' if o > 1 else 'steelblue' for o in odds_df['Odds Ratio']]
plt.barh(odds_df['Feature'], odds_df['Odds Ratio'], color=colors)
plt.axvline(x=1, color='black', linestyle='--', linewidth=1)  # line at 1 = no effect
plt.xlabel('Odds Ratio')
plt.title('Logistic Regression — Odds Ratios\n(Red = increases risk, Blue = decreases risk)')
plt.tight_layout()
plt.show()

---
## Section 6 — Random Forest *(Aryan Shah)*
Our second baseline. An ensemble of 200 decision trees that handles non-linear patterns
and automatically tells us which features matter most.

In [ ]:
# ── 6.1 Train Random Forest ──────────────────────────────────────────────────

# n_estimators=200 means we build 200 decision trees
# max_depth=10 limits how deep each tree can grow — prevents overfitting
# class_weight='balanced' same as Logistic Regression — pays more attention
#   to CHD=1 patients during training
# n_jobs=-1 uses all available CPU cores to train faster

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Train on the SMOTE-resampled training data
rf_model.fit(X_train_resampled, y_train_resampled)
print('Random Forest trained successfully.')

In [ ]:
# ── 6.2 Evaluate on Validation Set ───────────────────────────────────────────

rf_val_preds = rf_model.predict(X_val_scaled)
rf_val_proba = rf_model.predict_proba(X_val_scaled)[:, 1]

rf_val_accuracy  = accuracy_score(y_val, rf_val_preds)
rf_val_auc       = roc_auc_score(y_val, rf_val_proba)
rf_val_f1        = f1_score(y_val, rf_val_preds)
rf_val_recall    = recall_score(y_val, rf_val_preds)
rf_val_precision = precision_score(y_val, rf_val_preds)

print('=== Random Forest — Validation Set ===')
print(f'  Accuracy:  {rf_val_accuracy:.4f}')
print(f'  AUC-ROC:   {rf_val_auc:.4f}')
print(f'  F1 Score:  {rf_val_f1:.4f}')
print(f'  Recall:    {rf_val_recall:.4f}')
print(f'  Precision: {rf_val_precision:.4f}')
print()
print(classification_report(y_val, rf_val_preds, target_names=['No CHD', 'CHD']))

In [ ]:
# ── 6.3 Confusion Matrix ─────────────────────────────────────────────────────

cm_rf = confusion_matrix(y_val, rf_val_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Pred No CHD', 'Pred CHD'],
            yticklabels=['Actual No CHD', 'Actual CHD'])
plt.title('Random Forest — Confusion Matrix (Validation)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.4 ROC Curve ────────────────────────────────────────────────────────────

fpr_rf, tpr_rf, _ = roc_curve(y_val, rf_val_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr_rf, tpr_rf, color='green', lw=2,
         label=f'Random Forest (AUC = {rf_val_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Random Forest — ROC Curve (Validation)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.5 Feature Importances ──────────────────────────────────────────────────

# Random Forest automatically calculates feature importances
# This tells us how much each feature contributed to the predictions
# across all 200 trees combined
# Higher importance = more useful for predicting CHD

importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print('Random Forest — Feature Importances (sorted):')
print(feat_imp_df.to_string(index=False))

plt.figure(figsize=(8, 6))
sns.barplot(x='Importance', y='Feature', data=feat_imp_df, palette='Greens_r')
plt.title('Random Forest — Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## Section 7 — SVM with RBF Kernel *(Elyas Khoubach)*
Our comparison model. Uses a kernel trick to draw non-linear decision boundaries.
We tune C and gamma on the validation set using GridSearchCV.

In [ ]:
# ── 7.1 Hyperparameter Tuning with GridSearchCV ───────────────────────────────

# C controls the penalty for misclassifications during training
#   High C = model tries very hard to classify all training points correctly
#            (can overfit — learns the training data too specifically)
#   Low C  = model accepts some mistakes for a simpler, more general boundary
#
# gamma controls how far each data point's influence reaches
#   High gamma = each point only influences nearby points (complex boundary)
#   Low gamma  = each point influences a wider area (smoother boundary)
#
# GridSearchCV tries every combination of C and gamma
# and picks the one with the best AUC-ROC on the validation set

param_grid_svm = {
    'C':     [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.1]
}

# cv=5 means 5-fold cross-validation on the training set
# scoring='roc_auc' means we optimize for AUC-ROC
# probability=True is required to get probabilities for AUC-ROC calculation
svm_base = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)

svm_grid = GridSearchCV(
    svm_base,
    param_grid_svm,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# This will take a few minutes — it is trying 9 combinations × 5 folds = 45 fits
svm_grid.fit(X_train_resampled, y_train_resampled)

print(f'\nBest SVM parameters: {svm_grid.best_params_}')
print(f'Best cross-validation AUC-ROC: {svm_grid.best_score_:.4f}')

# Extract the best model
svm_model = svm_grid.best_estimator_

In [ ]:
# ── 7.2 Evaluate on Validation Set ───────────────────────────────────────────

svm_val_preds = svm_model.predict(X_val_scaled)
svm_val_proba = svm_model.predict_proba(X_val_scaled)[:, 1]

svm_val_accuracy  = accuracy_score(y_val, svm_val_preds)
svm_val_auc       = roc_auc_score(y_val, svm_val_proba)
svm_val_f1        = f1_score(y_val, svm_val_preds)
svm_val_recall    = recall_score(y_val, svm_val_preds)
svm_val_precision = precision_score(y_val, svm_val_preds)

print('=== SVM — Validation Set ===')
print(f'  Accuracy:  {svm_val_accuracy:.4f}')
print(f'  AUC-ROC:   {svm_val_auc:.4f}')
print(f'  F1 Score:  {svm_val_f1:.4f}')
print(f'  Recall:    {svm_val_recall:.4f}')
print(f'  Precision: {svm_val_precision:.4f}')
print()
print(classification_report(y_val, svm_val_preds, target_names=['No CHD', 'CHD']))

In [ ]:
# ── 7.3 Confusion Matrix ─────────────────────────────────────────────────────

cm_svm = confusion_matrix(y_val, svm_val_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Pred No CHD', 'Pred CHD'],
            yticklabels=['Actual No CHD', 'Actual CHD'])
plt.title('SVM — Confusion Matrix (Validation)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.4 ROC Curve ────────────────────────────────────────────────────────────

fpr_svm, tpr_svm, _ = roc_curve(y_val, svm_val_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr_svm, tpr_svm, color='purple', lw=2,
         label=f'SVM (AUC = {svm_val_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('SVM — ROC Curve (Validation)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Section 8 — XGBoost *(Matias Elban Stringa)*
Our refined solution. Gradient boosting with built-in regularization, native missing
value handling, hyperparameter tuning with early stopping, and SHAP explainability.

In [ ]:
# ── 8.1 Set Up Class Weight for XGBoost ──────────────────────────────────────

# XGBoost uses scale_pos_weight instead of class_weight='balanced'
# scale_pos_weight = number of negative cases / number of positive cases
# This tells XGBoost to treat each positive (CHD=1) case as if it were
# scale_pos_weight times more important than a negative case

neg_count = (y_train == 0).sum()   # number of No-CHD patients in training
pos_count = (y_train == 1).sum()   # number of CHD patients in training
scale_pos = neg_count / pos_count

print(f'Negative cases: {neg_count}')
print(f'Positive cases: {pos_count}')
print(f'scale_pos_weight = {scale_pos:.2f}')

In [ ]:
# ── 8.2 Train XGBoost with Early Stopping ────────────────────────────────────

# XGBoost hyperparameters explained:
# n_estimators=500     — maximum number of trees to build
# learning_rate=0.05   — how much each tree corrects the previous ones
#                        smaller = slower but more precise learning
# max_depth=6          — how deep each tree can grow
# subsample=0.8        — each tree only sees 80% of training data
#                        (adds randomness, prevents overfitting)
# colsample_bytree=0.8 — each tree only sees 80% of features
#                        (same idea — more randomness = better generalization)
# reg_alpha=0.1        — L1 regularization (makes some feature weights exactly 0)
# reg_lambda=1.0       — L2 regularization (shrinks all feature weights)
# scale_pos_weight     — handles class imbalance as calculated above
# eval_metric='auc'    — what to track during training for early stopping
# early_stopping_rounds=30 — stop training if AUC hasn't improved in 30 rounds

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos,
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42,
    use_label_encoder=False
)

# eval_set tells XGBoost to monitor performance on the validation set
# verbose=50 means print a progress update every 50 trees
xgb_model.fit(
    X_train_resampled, y_train_resampled,
    eval_set=[(X_val_scaled, y_val)],
    verbose=50
)

print(f'\nBest number of trees: {xgb_model.best_iteration}')
print(f'Best validation AUC:  {xgb_model.best_score:.4f}')

In [ ]:
# ── 8.3 Hyperparameter Tuning with GridSearchCV ───────────────────────────────

# After the initial model, we tune the most important hyperparameters
# to squeeze out better performance
# We search over learning_rate and max_depth combinations

param_grid_xgb = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth':     [4, 6, 8],
}

xgb_base = xgb.XGBClassifier(
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos,
    eval_metric='auc',
    random_state=42,
    use_label_encoder=False
)

xgb_grid = GridSearchCV(
    xgb_base,
    param_grid_xgb,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train_resampled, y_train_resampled)

print(f'\nBest XGBoost parameters: {xgb_grid.best_params_}')
print(f'Best cross-validation AUC-ROC: {xgb_grid.best_score_:.4f}')

# Use the best model going forward
xgb_model = xgb_grid.best_estimator_

In [ ]:
# ── 8.4 Evaluate on Validation Set ───────────────────────────────────────────

xgb_val_preds = xgb_model.predict(X_val_scaled)
xgb_val_proba = xgb_model.predict_proba(X_val_scaled)[:, 1]

xgb_val_accuracy  = accuracy_score(y_val, xgb_val_preds)
xgb_val_auc       = roc_auc_score(y_val, xgb_val_proba)
xgb_val_f1        = f1_score(y_val, xgb_val_preds)
xgb_val_recall    = recall_score(y_val, xgb_val_preds)
xgb_val_precision = precision_score(y_val, xgb_val_preds)

print('=== XGBoost — Validation Set ===')
print(f'  Accuracy:  {xgb_val_accuracy:.4f}')
print(f'  AUC-ROC:   {xgb_val_auc:.4f}')
print(f'  F1 Score:  {xgb_val_f1:.4f}')
print(f'  Recall:    {xgb_val_recall:.4f}')
print(f'  Precision: {xgb_val_precision:.4f}')
print()
print(classification_report(y_val, xgb_val_preds, target_names=['No CHD', 'CHD']))

In [ ]:
# ── 8.5 Confusion Matrix ─────────────────────────────────────────────────────

cm_xgb = confusion_matrix(y_val, xgb_val_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Pred No CHD', 'Pred CHD'],
            yticklabels=['Actual No CHD', 'Actual CHD'])
plt.title('XGBoost — Confusion Matrix (Validation)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.6 ROC Curve ────────────────────────────────────────────────────────────

fpr_xgb, tpr_xgb, _ = roc_curve(y_val, xgb_val_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr_xgb, tpr_xgb, color='darkorange', lw=2,
         label=f'XGBoost (AUC = {xgb_val_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('XGBoost — ROC Curve (Validation)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.7 SHAP Values ──────────────────────────────────────────────────────────

# SHAP explains WHY XGBoost made each individual prediction
# For every patient, SHAP assigns a score to each feature:
#   Positive SHAP value = this feature INCREASED the predicted CHD risk
#   Negative SHAP value = this feature DECREASED the predicted CHD risk
#
# TreeExplainer is the fastest SHAP method for tree-based models like XGBoost

explainer = shap.TreeExplainer(xgb_model)

# Calculate SHAP values for the validation set
# shap_values will have shape (number of patients, number of features)
shap_values = explainer.shap_values(X_val_scaled)

print('SHAP values calculated for', len(shap_values), 'patients')
print('Shape of SHAP values array:', shap_values.shape)

In [ ]:
# ── 8.8 SHAP Summary Plot ────────────────────────────────────────────────────

# This plot shows all features ranked by overall importance
# Each dot is one patient
# Color = feature value (red = high value, blue = low value)
# X-axis = SHAP value (how much this feature pushed the prediction)
# Features at the top have the biggest impact on predictions overall

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_val_scaled,
    feature_names=feature_names,
    show=False
)
plt.title('XGBoost — SHAP Summary Plot\n(Each dot = one patient, color = feature value)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.9 SHAP Bar Plot (Global Feature Importance) ────────────────────────────

# This simpler version shows the average absolute SHAP value per feature
# It answers: which features matter most overall across all patients?

shap.summary_plot(
    shap_values,
    X_val_scaled,
    feature_names=feature_names,
    plot_type='bar',
    show=False
)
plt.title('XGBoost — SHAP Feature Importance (Global)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.10 SHAP Waterfall — Explain One Individual Patient ─────────────────────

# This explains a single patient's prediction in detail
# It shows exactly which features pushed the risk up or down for that one person
# We look at the first patient in the validation set who actually has CHD

# Find the index of the first actual CHD patient in validation set
y_val_reset = y_val.reset_index(drop=True)
first_chd_idx = y_val_reset[y_val_reset == 1].index[0]

print(f'Explaining prediction for validation patient #{first_chd_idx}')
print(f'Actual label: {"CHD" if y_val_reset[first_chd_idx] == 1 else "No CHD"}')
print(f'Predicted probability of CHD: {xgb_val_proba[first_chd_idx]:.3f}')

# Waterfall plot for this specific patient
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[first_chd_idx],
        base_values=explainer.expected_value,
        data=X_val_scaled[first_chd_idx],
        feature_names=feature_names
    )
)

---
## Section 9 — Final Model Comparison
We now evaluate all four models on the **held-out test set** — the data none of the
models have seen during training or tuning. This gives us the honest final performance.

In [ ]:
# ── 9.1 Evaluate All Four Models on Test Set ──────────────────────────────────

# We now run each model on the test set for the first and only time
# These numbers are the real, honest performance numbers

def evaluate_on_test(model, X_test, y_test, model_name):
    """Run a trained model on the test set and return all metrics."""
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    return {
        'Model':     model_name,
        'Accuracy':  round(accuracy_score(y_test, preds), 4),
        'AUC-ROC':   round(roc_auc_score(y_test, proba), 4),
        'F1 Score':  round(f1_score(y_test, preds), 4),
        'Recall':    round(recall_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds), 4)
    }

# Run each model
results = [
    evaluate_on_test(lr_model,  X_test_scaled, y_test, 'Logistic Regression'),
    evaluate_on_test(rf_model,  X_test_scaled, y_test, 'Random Forest'),
    evaluate_on_test(svm_model, X_test_scaled, y_test, 'SVM'),
    evaluate_on_test(xgb_model, X_test_scaled, y_test, 'XGBoost'),
]

# Build the comparison table
results_df = pd.DataFrame(results).set_index('Model')

print('=' * 65)
print('       FINAL MODEL COMPARISON — TEST SET')
print('=' * 65)
print(results_df.to_string())
print('=' * 65)
print('\nBest model by AUC-ROC:', results_df['AUC-ROC'].idxmax())
print('Best model by Recall: ', results_df['Recall'].idxmax())

In [ ]:
# ── 9.2 Comparison Bar Chart ──────────────────────────────────────────────────

# Visualize all metrics side by side for easy comparison

metrics = ['Accuracy', 'AUC-ROC', 'F1 Score', 'Recall', 'Precision']
colors  = ['steelblue', 'green', 'purple', 'darkorange']
models  = results_df.index.tolist()

fig, axes = plt.subplots(1, len(metrics), figsize=(18, 5))

for ax, metric in zip(axes, metrics):
    values = results_df[metric].values
    bars = ax.bar(models, values, color=colors, alpha=0.85, edgecolor='white')
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
    ax.set_xticklabels(models, rotation=20, ha='right', fontsize=9)
    # Add value labels on top of each bar
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Final Model Comparison — Test Set', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.3 All ROC Curves on One Plot ───────────────────────────────────────────

# This puts all four ROC curves on the same axes so we can compare them visually
# A curve that goes higher and further left is better

plt.figure(figsize=(8, 6))

model_configs = [
    (lr_model,  'steelblue',  'Logistic Regression'),
    (rf_model,  'green',      'Random Forest'),
    (svm_model, 'purple',     'SVM'),
    (xgb_model, 'darkorange', 'XGBoost'),
]

for model, color, name in model_configs:
    proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('All Models — ROC Curves (Test Set)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.4 Final Summary ─────────────────────────────────────────────────────────

print('FINAL RESULTS SUMMARY')
print('=' * 65)
print(results_df.to_string())
print('=' * 65)

best_auc_model    = results_df['AUC-ROC'].idxmax()
best_recall_model = results_df['Recall'].idxmax()
best_auc_val      = results_df['AUC-ROC'].max()
best_recall_val   = results_df['Recall'].max()

print(f'\nBest AUC-ROC:  {best_auc_model} ({best_auc_val:.4f})')
print(f'Best Recall:   {best_recall_model} ({best_recall_val:.4f})')
print()
print('Targets from proposal:')
print(f'  AUC-ROC > 0.85:    {"MET" if best_auc_val > 0.85 else "NOT MET"} ({best_auc_val:.4f})')
print(f'  Recall  > 0.80:    {"MET" if best_recall_val > 0.80 else "NOT MET"} ({best_recall_val:.4f})')